# Audio Embedding Dataset Generator for ESP32-S3 Phase 3

Generate a large audio-text dataset (~32k samples) for training audio embeddings.

## Auto-Parallelized across 12 GPUs

This notebook automatically distributes work across all available GPUs using multiprocessing.
No manual configuration needed - just run all cells!

## Pipeline
1. Load 2008 questions from `full.csv`
2. Cluster similar questions into buckets for contrastive learning
3. Generate audio with 8 TED voices (XTTS-v2) - **AUTO-PARALLELIZED across 12 GPUs**
4. Apply audio augmentation (gaussian noise, pitch, speed)
5. Extract YAMNet embeddings
6. Train contrastive learning model (with bucketed sampling)
7. Export TFLite for ESP32

## Output
- `audio_projection_quantized.tflite` (~663KB)
- `embeddings.bin` (2008 x 256-dim)
- `intents.txt` (2008 question strings)

---

## 1. Configuration

In [28]:
#=============================================================================
# CONFIGURATION
#=============================================================================

import torch

# Auto-detect available GPUs
NUM_GPUS = torch.cuda.device_count()
print(f"Detected {NUM_GPUS} GPUs")

# Processing
VOICES_PER_TEXT = 8          # 8 TED speakers
AUGMENTATIONS_PER_AUDIO = 2  # Original + 1 augmented (with gaussian noise)

# Bucketing for contrastive learning
SIMILARITY_THRESHOLD = 0.85  # Questions above this similarity are in same bucket
MIN_BUCKET_SIZE = 1
MAX_BUCKET_SIZE = 10

# Embedding dimensions
YAMNET_DIM = 1024            # YAMNet output dimension
TEXT_DIM = 768               # MPNet output dimension
EMBEDDING_DIM = 256          # Final projection dimension

# Training
BATCH_SIZE = 32
EPOCHS = 3000
LEARNING_RATE = 1.5e-4
TEMPERATURE = 0.2

# Paths
from pathlib import Path
WORK_DIR = Path('/workspace') if Path('/workspace').exists() else Path('.')
DATA_DIR = WORK_DIR / 'data' / 'raw'
SPEAKER_DIR = WORK_DIR / 'speaker_voices'
AUDIO_DIR = WORK_DIR / 'audio_data'
MODEL_DIR = WORK_DIR / 'models'
LOG_DIR = WORK_DIR / 'logs'

# Suppress warnings
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='torchaudio')
warnings.filterwarnings('ignore', category=FutureWarning)

print(f"\nConfiguration:")
print(f"  Work directory: {WORK_DIR}")
print(f"  GPUs available: {NUM_GPUS}")
print(f"  Voices per text: {VOICES_PER_TEXT}")
print(f"  Augmentations: {AUGMENTATIONS_PER_AUDIO}")
print(f"\nExpected dataset size:")
print(f"  Base: 2008 questions")
print(f"  x {VOICES_PER_TEXT} voices = {2008 * VOICES_PER_TEXT:,}")
print(f"  x {AUGMENTATIONS_PER_AUDIO} aug = {2008 * VOICES_PER_TEXT * AUGMENTATIONS_PER_AUDIO:,}")
print(f"\nWork will be split across {NUM_GPUS} GPUs (~{2008 // NUM_GPUS} questions each)")

Detected 12 GPUs

Configuration:
  Work directory: /workspace
  GPUs available: 12
  Voices per text: 8
  Augmentations: 2

Expected dataset size:
  Base: 2008 questions
  x 8 voices = 16,064
  x 2 aug = 32,128

Work will be split across 12 GPUs (~167 questions each)


## 2. Environment Setup

In [2]:
# Install dependencies (vast.ai compatible with uv)
!uv add torch torchaudio coqui-tts tensorflow tensorflow-hub librosa soundfile audiomentations sentence-transformers pandas numpy matplotlib tqdm scikit-learn
print("Dependencies installed!")

Resolved 211 packages in 0.83ms
Audited 203 packages in 2ms
Dependencies installed!


In [3]:
import torch
import tensorflow as tf
import numpy as np
import pandas as pd
import librosa
import soundfile as sf
from pathlib import Path
import os
import random
import json
import struct
from datetime import datetime
from glob import glob
from tqdm import tqdm
import urllib.request
import matplotlib.pyplot as plt
from collections import defaultdict

print(f"PyTorch: {torch.__version__}")
print(f"TensorFlow: {tf.__version__}")
print(f"GPU (PyTorch): {torch.cuda.is_available()}")
print(f"GPU (TF): {tf.config.list_physical_devices('GPU')}")

2025-11-28 11:46:47.506224: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


PyTorch: 2.8.0+cu128
TensorFlow: 2.20.0
GPU (PyTorch): True
GPU (TF): [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:2', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:3', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:4', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:5', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:6', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:7', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:8', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:9', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:10', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:11', device_type='GPU')]


In [4]:
# Create directory structure
for d in [DATA_DIR, SPEAKER_DIR, AUDIO_DIR, MODEL_DIR, LOG_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Directory structure created at {WORK_DIR}")

Directory structure created at /workspace


## 3. Load Dataset & Create Buckets

For contrastive learning to work properly:
- Similar questions (e.g., "What is the email" vs "Tell me the email address") should be in the SAME bucket
- During training, we sample ONE audio per bucket per batch
- This ensures negatives are truly different questions, not just audio variations of the same question

In [5]:
from sentence_transformers import SentenceTransformer
from sklearn.cluster import AgglomerativeClustering
from sklearn.metrics.pairwise import cosine_similarity

# Load text encoder for bucketing
print("Loading MiniLM for question bucketing...")
text_encoder = SentenceTransformer('sentence-transformers/all-mpnet-base-v2')

# Load questions
qa_file = DATA_DIR / 'full.csv'
if not qa_file.exists():
    raise FileNotFoundError(f"full.csv not found at {qa_file}")

qa_df = pd.read_csv(qa_file)
questions = qa_df['question'].tolist()
answers = qa_df['answer'].tolist() if 'answer' in qa_df.columns else [None] * len(questions)

print(f"Loaded {len(questions)} questions")

Loading MiniLM for question bucketing...
Loaded 2008 questions


In [6]:
# Encode all questions for clustering
print("Encoding questions for clustering...")
question_embeddings = text_encoder.encode(questions, show_progress_bar=True)
print(f"Question embeddings shape: {question_embeddings.shape}")

Encoding questions for clustering...


Batches:   0%|          | 0/63 [00:00<?, ?it/s]

Question embeddings shape: (2008, 768)


In [7]:
# Cluster similar questions into buckets
print(f"Clustering questions (similarity threshold: {SIMILARITY_THRESHOLD})...")

# Use Agglomerative Clustering with cosine distance
distance_threshold = 1 - SIMILARITY_THRESHOLD  # Convert similarity to distance
clustering = AgglomerativeClustering(
    n_clusters=None,
    distance_threshold=distance_threshold,
    metric='cosine',
    linkage='average'
)
cluster_labels = clustering.fit_predict(question_embeddings)

# Create buckets
buckets = defaultdict(list)
for idx, label in enumerate(cluster_labels):
    buckets[label].append({
        'idx': idx,
        'question': questions[idx],
        'answer': answers[idx],
        'embedding': question_embeddings[idx]
    })

num_buckets = len(buckets)
bucket_sizes = [len(b) for b in buckets.values()]

print(f"\nBucketing Results:")
print(f"  Total questions: {len(questions)}")
print(f"  Number of buckets: {num_buckets}")
print(f"  Avg bucket size: {np.mean(bucket_sizes):.1f}")
print(f"  Min bucket size: {min(bucket_sizes)}")
print(f"  Max bucket size: {max(bucket_sizes)}")

# Show sample buckets
print(f"\nSample buckets (showing first 3):")
for i, (label, items) in enumerate(list(buckets.items())[:3]):
    print(f"  Bucket {label} ({len(items)} questions):")
    for item in items[:3]:
        print(f"    - {item['question'][:60]}...")

Clustering questions (similarity threshold: 0.85)...

Bucketing Results:
  Total questions: 2008
  Number of buckets: 1698
  Avg bucket size: 1.2
  Min bucket size: 1
  Max bucket size: 8

Sample buckets (showing first 3):
  Bucket 149 (2 questions):
    - What is Alphons email address...
    - Tell me Alphons contact email...
  Bucket 1621 (1 questions):
    - What is his phone number...
  Bucket 179 (3 questions):
    - When was Alphons born...
    - Which month was Alphons born in...
    - What month is Alphons birthday...


In [8]:
# Save bucket mapping for reference
bucket_mapping = {
    'num_buckets': num_buckets,
    'questions_per_bucket': {str(k): len(v) for k, v in buckets.items()},
    'bucket_assignments': {str(i): int(cluster_labels[i]) for i in range(len(questions))}
}

with open(LOG_DIR / 'bucket_mapping.json', 'w') as f:
    json.dump(bucket_mapping, f, indent=2)

print(f"Bucket mapping saved to {LOG_DIR / 'bucket_mapping.json'}")

Bucket mapping saved to /workspace/logs/bucket_mapping.json


## 4. Download TED Speaker Samples

In [9]:
# TED speaker samples from audio-samples.github.io (diverse voices)
ted_speakers = [
    {'name': 'BillGates', 'url': 'https://raw.githubusercontent.com/audio-samples/audio-samples.github.io/master/samples/mp3/ted_speakers/BillGates/sample-0.mp3', 'desc': 'Male US'},
    {'name': 'DaphneKoller', 'url': 'https://raw.githubusercontent.com/audio-samples/audio-samples.github.io/master/samples/mp3/ted_speakers/DaphneKoller/sample-0.mp3', 'desc': 'Female US'},
    {'name': 'FeiFeiLi', 'url': 'https://raw.githubusercontent.com/audio-samples/audio-samples.github.io/master/samples/mp3/ted_speakers/FeiFeiLi/sample-0.mp3', 'desc': 'Female Chinese-American'},
    {'name': 'JaneGoodall', 'url': 'https://raw.githubusercontent.com/audio-samples/audio-samples.github.io/master/samples/mp3/ted_speakers/JaneGoodall/sample-1.mp3', 'desc': 'Female British'},
    {'name': 'SalmanKhan', 'url': 'https://raw.githubusercontent.com/audio-samples/audio-samples.github.io/master/samples/mp3/ted_speakers/SalmanKhan/sample-0.mp3', 'desc': 'Male US'},
    {'name': 'GeorgeTakei', 'url': 'https://raw.githubusercontent.com/audio-samples/audio-samples.github.io/master/samples/mp3/ted_speakers/GeorgeTakei/sample-0.mp3', 'desc': 'Male US distinctive'},
    {'name': 'StephenHawking', 'url': 'https://raw.githubusercontent.com/audio-samples/audio-samples.github.io/master/samples/mp3/ted_speakers/StephenHawking/sample-0.mp3', 'desc': 'Male British synthesized'},
    {'name': 'StephenWolfram', 'url': 'https://raw.githubusercontent.com/audio-samples/audio-samples.github.io/master/samples/mp3/ted_speakers/StephenWolfram/sample-0.mp3', 'desc': 'Male British'},
]

speaker_files = []
print("Downloading TED speaker samples...")

for speaker in ted_speakers:
    name = speaker['name']
    url = speaker['url']
    desc = speaker['desc']
    
    mp3_file = SPEAKER_DIR / f'ted_{name}.mp3'
    wav_file = SPEAKER_DIR / f'ted_{name}.wav'
    
    if not wav_file.exists():
        try:
            print(f"  Downloading {name} ({desc})...")
            urllib.request.urlretrieve(url, mp3_file)
            
            # Convert to WAV (16kHz mono, 6 sec for XTTS)
            audio, sr = librosa.load(mp3_file, sr=16000, mono=True)
            if len(audio) > 6 * 16000:
                audio = audio[:6 * 16000]
            sf.write(wav_file, audio, 16000)
            mp3_file.unlink()  # Remove MP3
            print(f"    Saved {name} (6 sec @ 16kHz)")
        except Exception as e:
            print(f"    Failed: {e}")
            continue
    else:
        print(f"  {name} already exists ({desc})")
    
    speaker_files.append(str(wav_file))

print(f"\nReady with {len(speaker_files)} TED speaker references")

  BillGates already exists (Male US)
  DaphneKoller already exists (Female US)
  FeiFeiLi already exists (Female Chinese-American)
  JaneGoodall already exists (Female British)
  SalmanKhan already exists (Male US)
  GeorgeTakei already exists (Male US distinctive)
  StephenHawking already exists (Male British synthesized)
  StephenWolfram already exists (Male British)

Ready with 8 TED speaker references


## 5. Load XTTS-v2 Model

In [10]:
# We'll load XTTS inside each worker process (one per GPU)
# Just verify GPU setup here

print(f"GPU Setup:")
for i in range(NUM_GPUS):
    props = torch.cuda.get_device_properties(i)
    print(f"  GPU {i}: {props.name} ({props.total_memory / 1024**3:.1f} GB)")

print(f"\nXTTS-v2 will be loaded separately on each GPU in parallel workers")

GPU Setup:
  GPU 0: NVIDIA GeForce RTX 4070 SUPER (11.6 GB)
  GPU 1: NVIDIA GeForce RTX 4070 SUPER (11.6 GB)
  GPU 2: NVIDIA GeForce RTX 4070 SUPER (11.6 GB)
  GPU 3: NVIDIA GeForce RTX 4070 SUPER (11.6 GB)
  GPU 4: NVIDIA GeForce RTX 4070 SUPER (11.6 GB)
  GPU 5: NVIDIA GeForce RTX 4070 SUPER (11.6 GB)
  GPU 6: NVIDIA GeForce RTX 4070 SUPER (11.6 GB)
  GPU 7: NVIDIA GeForce RTX 4070 SUPER (11.6 GB)
  GPU 8: NVIDIA GeForce RTX 4070 SUPER (11.6 GB)
  GPU 9: NVIDIA GeForce RTX 4070 SUPER (11.6 GB)
  GPU 10: NVIDIA GeForce RTX 4070 SUPER (11.6 GB)
  GPU 11: NVIDIA GeForce RTX 4070 SUPER (11.6 GB)

XTTS-v2 will be loaded separately on each GPU in parallel workers


## 6. Audio Generation Utilities

In [11]:
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift

# Audio augmentation pipeline (will be used by workers)
# Defined here so it's available globally

print("Audio augmentation pipeline configured")

Audio augmentation pipeline configured


## 7. State Persistence (Resume Capability)

In [12]:
# State tracking utilities (used by workers)
def get_state_file(worker_id):
    return LOG_DIR / f'generation_state_worker_{worker_id}.json'

def save_generation_state(worker_id, question_idx, speaker_idx, total_generated):
    """Save generation progress for resume"""
    import json
    from datetime import datetime
    state = {
        'worker_id': worker_id,
        'question_idx': question_idx,
        'speaker_idx': speaker_idx,
        'total_generated': total_generated,
        'timestamp': datetime.now().isoformat()
    }
    with open(get_state_file(worker_id), 'w') as f:
        json.dump(state, f, indent=2)

def load_generation_state(worker_id):
    """Load generation progress for resume"""
    import json
    state_file = get_state_file(worker_id)
    if state_file.exists():
        with open(state_file) as f:
            return json.load(f)
    return None

print("State persistence utilities defined")

State persistence utilities defined


## 8. Parallel Audio Generation (12 GPUs)

Each GPU runs its own worker process with:
- Dedicated XTTS-v2 model instance
- Pre-computed speaker latents
- Subset of questions to process

All 12 GPUs work simultaneously!

In [13]:
# Create output directories
for bucket_id in buckets.keys():
    (AUDIO_DIR / f'bucket_{bucket_id}').mkdir(parents=True, exist_ok=True)

# Prepare data for workers (must be picklable)
worker_data = {
    'questions': questions,
    'cluster_labels': cluster_labels.tolist(),
    'speaker_files': speaker_files,
    'num_gpus': NUM_GPUS,
    'audio_dir': str(AUDIO_DIR),
    'log_dir': str(LOG_DIR),
}

print(f"Prepared data for {NUM_GPUS} workers")
print(f"  Total questions: {len(questions)}")
print(f"  Questions per worker: ~{len(questions) // NUM_GPUS}")

Prepared data for 12 workers
  Total questions: 2008
  Questions per worker: ~167


In [14]:
# Write worker function to a separate file (required for multiprocessing in Jupyter)
worker_code = '''
import os
import json
import torch
import torchaudio
import librosa
import soundfile as sf
import numpy as np
from pathlib import Path
from datetime import datetime
from tqdm import tqdm
from audiomentations import Compose, AddGaussianNoise, TimeStretch, PitchShift
import warnings
warnings.filterwarnings('ignore', category=UserWarning, module='torchaudio')
warnings.filterwarnings('ignore', category=FutureWarning)

def gpu_worker(args):
    """Worker function that runs on a specific GPU"""
    worker_id, data = args
    
    # Set this worker to use specific GPU
    os.environ['CUDA_VISIBLE_DEVICES'] = str(worker_id)
    
    # Extract data
    questions = data['questions']
    cluster_labels = data['cluster_labels']
    speaker_files = data['speaker_files']
    num_gpus = data['num_gpus']
    audio_dir = Path(data['audio_dir'])
    log_dir = Path(data['log_dir'])
    
    # Calculate this worker's question range
    questions_per_worker = len(questions) // num_gpus
    start_idx = worker_id * questions_per_worker
    end_idx = start_idx + questions_per_worker if worker_id < num_gpus - 1 else len(questions)
    worker_questions = list(range(start_idx, end_idx))
    
    print(f"[GPU {worker_id}] Starting: questions {start_idx}-{end_idx-1} ({len(worker_questions)} questions)")
    
    # Load XTTS on this GPU
    from TTS.api import TTS
    tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2").to("cuda")
    
    # Pre-compute speaker latents
    print(f"[GPU {worker_id}] Pre-computing speaker latents...")
    speaker_latents = {}
    for sf_path in speaker_files:
        gpt_cond_latent, speaker_embedding = tts.synthesizer.tts_model.get_conditioning_latents(
            audio_path=sf_path
        )
        speaker_latents[sf_path] = (gpt_cond_latent, speaker_embedding)
    
    # Setup augmentation
    augmenter = Compose([
        AddGaussianNoise(min_amplitude=0.005, max_amplitude=0.02, p=1.0),
        TimeStretch(min_rate=0.9, max_rate=1.1, p=0.5),
        PitchShift(min_semitones=-2, max_semitones=2, p=0.5),
    ])
    
    # Check for resume
    state_file = log_dir / f'generation_state_worker_{worker_id}.json'
    if state_file.exists():
        with open(state_file) as f:
            state = json.load(f)
        resume_q_idx = state['question_idx']
        resume_s_idx = state['speaker_idx']
        total_generated = state['total_generated']
        print(f"[GPU {worker_id}] Resuming from q{resume_q_idx}, s{resume_s_idx}")
    else:
        resume_q_idx = start_idx
        resume_s_idx = 0
        total_generated = 0
    
    metadata = []
    
    # Process questions
    for q_idx in tqdm(worker_questions, desc=f"GPU {worker_id}", position=worker_id, leave=True):
        if q_idx < resume_q_idx:
            continue
        
        question = questions[q_idx]
        bucket_id = int(cluster_labels[q_idx])
        output_dir = audio_dir / f'bucket_{bucket_id}'
        output_dir.mkdir(parents=True, exist_ok=True)
        
        for s_idx, speaker_file in enumerate(speaker_files):
            if q_idx == resume_q_idx and s_idx < resume_s_idx:
                continue
            
            base_name = f'q{q_idx:04d}_s{s_idx}'
            audio_file = output_dir / f'{base_name}.wav'
            
            if not audio_file.exists():
                try:
                    # Generate audio
                    gpt_cond_latent, speaker_embedding = speaker_latents[speaker_file]
                    out = tts.synthesizer.tts_model.inference(
                        question, "en", gpt_cond_latent, speaker_embedding
                    )
                    wav_tensor = torch.tensor(out["wav"]).unsqueeze(0)
                    torchaudio.save(str(audio_file), wav_tensor, 24000)
                    
                    # Load and create augmented version
                    audio, sr = librosa.load(str(audio_file), sr=16000, mono=True)
                    
                    metadata.append({
                        'file': str(audio_file),
                        'text': question,
                        'question_idx': q_idx,
                        'bucket_id': bucket_id,
                        'speaker': s_idx,
                        'variant': 'original',
                        'worker_id': worker_id
                    })
                    
                    # Augmented version
                    audio_aug = augmenter(samples=audio, sample_rate=sr)
                    aug_file = output_dir / f'{base_name}_aug.wav'
                    sf.write(str(aug_file), audio_aug, sr)
                    
                    metadata.append({
                        'file': str(aug_file),
                        'text': question,
                        'question_idx': q_idx,
                        'bucket_id': bucket_id,
                        'speaker': s_idx,
                        'variant': 'augmented',
                        'worker_id': worker_id
                    })
                    
                    total_generated += 2
                except Exception as e:
                    print(f"[GPU {worker_id}] Error {base_name}: {e}")
                    continue
            
            # Save state periodically
            if total_generated % 50 == 0 and total_generated > 0:
                state = {
                    'worker_id': worker_id,
                    'question_idx': q_idx,
                    'speaker_idx': s_idx,
                    'total_generated': total_generated,
                    'timestamp': datetime.now().isoformat()
                }
                with open(state_file, 'w') as f:
                    json.dump(state, f, indent=2)
    
    # Save metadata
    metadata_file = audio_dir / f'metadata_worker_{worker_id}.json'
    with open(metadata_file, 'w') as f:
        json.dump(metadata, f, indent=2)
    
    print(f"[GPU {worker_id}] DONE! Generated {total_generated} files")
    return worker_id, total_generated
'''

# Write to file
worker_file = WORK_DIR / 'gpu_worker.py'
with open(worker_file, 'w') as f:
    f.write(worker_code)

print(f"Worker module written to: {worker_file}")

Worker module written to: /workspace/gpu_worker.py


In [ ]:
# Launch all GPU workers in parallel
import sys
import multiprocessing as mp

# Add workspace to path and import worker
sys.path.insert(0, str(WORK_DIR))
from gpu_worker import gpu_worker

print(f"="*60)
print(f"LAUNCHING {NUM_GPUS} PARALLEL GPU WORKERS")
print(f"="*60)
print(f"Each GPU will process ~{len(questions) // NUM_GPUS} questions")
print(f"Total audio files to generate: {len(questions) * len(speaker_files) * 2:,}")
print()

# Prepare arguments for each worker
worker_args = [(i, worker_data) for i in range(NUM_GPUS)]

# Use spawn method for CUDA compatibility
ctx = mp.get_context('spawn')
with ctx.Pool(processes=NUM_GPUS) as pool:
    results = pool.map(gpu_worker, worker_args)

# Summary
print(f"\n{'='*60}")
print(f"ALL WORKERS COMPLETE!")
print(f"{'='*60}")
total_files = sum(r[1] for r in results)
print(f"Total files generated: {total_files:,}")
for worker_id, count in results:
    print(f"  GPU {worker_id}: {count:,} files")

LAUNCHING 12 PARALLEL GPU WORKERS
Each GPU will process ~167 questions
Total audio files to generate: 32,128

[GPU 11] Starting: questions 1837-2007 (171 questions)
[GPU 3] Starting: questions 501-667 (167 questions)
[GPU 8] Starting: questions 1336-1502 (167 questions)
[GPU 6] Starting: questions 1002-1168 (167 questions)
[GPU 0] Starting: questions 0-166 (167 questions)
[GPU 2] Starting: questions 334-500 (167 questions)
[GPU 10] Starting: questions 1670-1836 (167 questions)
[GPU 7] Starting: questions 1169-1335 (167 questions)
[GPU 4] Starting: questions 668-834 (167 questions)
[GPU 9] Starting: questions 1503-1669 (167 questions)
[GPU 5] Starting: questions 835-1001 (167 questions)
[GPU 1] Starting: questions 167-333 (167 questions)


2025-11-28 11:47:03.702683: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-28 11:47:03.898569: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-28 11:47:03.928847: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-11-28 11:47:03.955980: I tensorflow/core/platform/cpu_featu

[GPU 11] Pre-computing speaker latents...













GPU 11:   0%|          | 0/171 [00:00<?, ?it/s]

[GPU 3] Pre-computing speaker latents...
[GPU 6] Pre-computing speaker latents...
[GPU 0] Pre-computing speaker latents...
[GPU 8] Pre-computing speaker latents...
[GPU 1] Pre-computing speaker latents...
[GPU 4] Pre-computing speaker latents...
[GPU 2] Pre-computing speaker latents...
[GPU 7] Pre-computing speaker latents...
[GPU 5] Pre-computing speaker latents...
[GPU 9] Pre-computing speaker latents...
[GPU 10] Pre-computing speaker latents...


...









GPU 8:  83%|████████▎ | 138/167 [18:45<03:51,  7.97s/it]



GPU 4:  73%|███████▎  | 122/167 [18:44<06:36,  8.81s/it]

GPU 2:  72%|███████▏  | 120/167 [18:43<07:33,  9.65s/it]










GPU 11:  82%|████████▏ | 140/171 [19:24<03:54,  7.55s/it]


GPU 3:  60%|█████▉    | 100/167 [18:51<11:59, 10.74s/it]





GPU 6:  77%|███████▋  | 128/167 [18:50<05:24,  8.32s/it]






GPU 7:  71%|███████▏  | 119/167 [18:46<07:52,  9.85s/it]








GPU 9:  81%|████████  | 135/167 [18:47<04:05,  7.66s/it]







GPU 1:  69%|██████▉   | 116/167 [18:51<07:24,  8.72s/it]




GPU 5:  77%|███████▋  | 128/167 [18:48<06:23,  9.84s/it]









GPU 10:  81%|████████▏ | 136/167 [18:48<04:22,  8.48s/it]


GPU 3:  60%|██████    | 101/167 [18:58<10:43,  9.76s/it]










GPU 11:  82%|████████▏ | 141/171 [19:33<03:53,  7.79s/it]

GPU 2:  72%|███████▏  | 121/167 [18:52<07:17,  9.50s/it]





GPU 6:  77%|███████▋  | 129/167 [18:58<05:16,  8.33s/it]






GPU 7:  72%|███████▏  | 120/167 [18:54<07:06,  9.08s/it

[GPU 8] DONE! Generated 2672 files




GPU 2:  87%|████████▋ | 145/167 [22:44<03:46, 10.32s/it]




GPU 5:  92%|█████████▏| 154/167 [22:45<01:54,  8.82s/it]










GPU 11: 100%|██████████| 171/171 [23:28<00:00,  8.24s/it]


[GPU 11] DONE! Generated 2736 files












GPU 10:  99%|█████████▉| 166/167 [22:48<00:08,  8.04s/it]





GPU 1:  85%|████████▌ | 142/167 [22:53<04:14, 10.17s/it]



GPU 4:  91%|█████████ | 152/167 [22:52<02:23,  9.57s/it]








GPU 9:  98%|█████████▊| 164/167 [22:51<00:28,  9.50s/it]






GPU 7:  89%|████████▊ | 148/167 [22:52<02:56,  9.31s/it]

GPU 2:  87%|████████▋ | 146/167 [22:53<03:26,  9.84s/it]




GPU 5:  93%|█████████▎| 155/167 [22:54<01:45,  8.76s/it]

## 9. Load Pre-trained Models (YAMNet)

In [ ]:
import tensorflow_hub as hub

print("Loading YAMNet...")
yamnet_model = hub.load('https://tfhub.dev/google/yamnet/1')
print("YAMNet loaded!")

In [ ]:
# Merge metadata from all GPU workers and extract embeddings

def load_audio_for_yamnet(file_path, target_sr=16000, target_length=3):
    """Load and prepare audio for YAMNet (3 seconds, 16kHz)"""
    audio, sr = librosa.load(file_path, sr=target_sr, mono=True)
    
    # Pad or trim to target length
    target_samples = target_length * target_sr
    if len(audio) < target_samples:
        audio = np.pad(audio, (0, target_samples - len(audio)))
    else:
        audio = audio[:target_samples]
    
    return audio.astype(np.float32)

# Merge metadata from all GPU workers
print("Merging metadata from all GPU workers...")
metadata = []
for worker_id in range(NUM_GPUS):
    worker_metadata_file = AUDIO_DIR / f'metadata_worker_{worker_id}.json'
    if worker_metadata_file.exists():
        with open(worker_metadata_file) as f:
            worker_data = json.load(f)
            metadata.extend(worker_data)
            print(f"  GPU {worker_id}: {len(worker_data)} entries")
    else:
        print(f"  GPU {worker_id}: NOT FOUND!")

print(f"\nTotal metadata entries: {len(metadata)}")

if len(metadata) == 0:
    raise RuntimeError("No metadata found! Check worker outputs.")

# Save merged metadata
merged_metadata_file = AUDIO_DIR / 'metadata.json'
with open(merged_metadata_file, 'w') as f:
    json.dump(metadata, f, indent=2)
print(f"Merged metadata saved: {merged_metadata_file}")

print(f"\nExtracting YAMNet embeddings...")
print(f"Total files: {len(metadata)}")

audio_embeddings = []
valid_metadata = []

for item in tqdm(metadata):
    file_path = item['file']
    
    if not Path(file_path).exists():
        continue
    
    try:
        audio = load_audio_for_yamnet(file_path)
        _, embeddings, _ = yamnet_model(audio)
        avg_embedding = tf.reduce_mean(embeddings, axis=0).numpy()
        
        audio_embeddings.append(avg_embedding)
        valid_metadata.append(item)
    except Exception as e:
        print(f"\nError processing {file_path}: {e}")
        continue

audio_embeddings = np.array(audio_embeddings)
print(f"\nYAMNet embeddings shape: {audio_embeddings.shape}")
print(f"Valid samples: {len(valid_metadata)}")

## 10. Extract YAMNet Embeddings

Merge metadata from all GPU workers and extract audio embeddings.

## 11. Prepare Text Embeddings (Per Bucket)

In [ ]:
# Get text embeddings for each audio sample
# Key: use the bucket's representative embedding for contrastive learning

# Create bucket -> representative embedding mapping
# Use mean of all question embeddings in each bucket
bucket_embeddings = {}
for bucket_id, items in buckets.items():
    bucket_embs = np.array([item['embedding'] for item in items])
    bucket_embeddings[bucket_id] = np.mean(bucket_embs, axis=0)

# Get text embedding for each audio sample based on its bucket
text_embeddings = []
bucket_ids = []

for item in valid_metadata:
    bucket_id = item['bucket_id']
    text_embeddings.append(bucket_embeddings[bucket_id])
    bucket_ids.append(bucket_id)

text_embeddings = np.array(text_embeddings)
bucket_ids = np.array(bucket_ids)

print(f"Text embeddings shape: {text_embeddings.shape}")
print(f"Unique buckets in training data: {len(np.unique(bucket_ids))}")

## 12. Build Projection Models

In [ ]:
# Audio projection: 1024 -> 256
audio_projection = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(YAMNET_DIM,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(512, activation='relu'),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.2),
    tf.keras.layers.Dense(EMBEDDING_DIM),
    tf.keras.layers.Lambda(lambda x: tf.nn.l2_normalize(x, axis=1))
], name='audio_projection')

# Text projection: 384 -> 256
text_projection = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(TEXT_DIM,)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dense(EMBEDDING_DIM),
    tf.keras.layers.Lambda(lambda x: tf.nn.l2_normalize(x, axis=1))
], name='text_projection')

print("Audio Projection Model:")
audio_projection.summary()
print("\nText Projection Model:")
text_projection.summary()

## 13. Bucketed Batch Sampler for Contrastive Learning

**Key insight**: For contrastive loss to work, each batch must have:
- ONE audio sample per bucket (no duplicates from same bucket)
- This ensures all other samples in the batch are true negatives

In [ ]:
class BucketedBatchGenerator:
    """Generate batches where each sample is from a different bucket.
    
    This ensures proper contrastive learning where negatives are truly
    from different questions, not just different audio of the same question.
    """
    
    def __init__(self, audio_embs, text_embs, bucket_ids, batch_size):
        self.audio_embs = audio_embs
        self.text_embs = text_embs
        self.bucket_ids = bucket_ids
        self.batch_size = batch_size
        
        # Group samples by bucket
        self.bucket_to_indices = defaultdict(list)
        for idx, bid in enumerate(bucket_ids):
            self.bucket_to_indices[bid].append(idx)
        
        self.all_buckets = list(self.bucket_to_indices.keys())
        
        # Ensure batch_size <= num_buckets
        if batch_size > len(self.all_buckets):
            print(f"Warning: batch_size ({batch_size}) > num_buckets ({len(self.all_buckets)})")
            print(f"Reducing batch_size to {len(self.all_buckets)}")
            self.batch_size = len(self.all_buckets)
    
    def __len__(self):
        return len(self.audio_embs) // self.batch_size
    
    def generate_batch(self):
        """Generate one batch with one sample per bucket."""
        # Randomly select batch_size buckets
        selected_buckets = random.sample(self.all_buckets, self.batch_size)
        
        batch_audio = []
        batch_text = []
        
        for bucket_id in selected_buckets:
            # Randomly select one sample from this bucket
            idx = random.choice(self.bucket_to_indices[bucket_id])
            batch_audio.append(self.audio_embs[idx])
            batch_text.append(self.text_embs[idx])
        
        return np.array(batch_audio), np.array(batch_text)
    
    def generate_epoch(self):
        """Generate batches for one epoch."""
        for _ in range(len(self)):
            yield self.generate_batch()

# Create batch generator
batch_generator = BucketedBatchGenerator(
    audio_embeddings,
    text_embeddings,
    bucket_ids,
    BATCH_SIZE
)

print(f"Bucketed Batch Generator:")
print(f"  Samples: {len(audio_embeddings)}")
print(f"  Buckets: {len(batch_generator.all_buckets)}")
print(f"  Batch size: {batch_generator.batch_size}")
print(f"  Batches per epoch: {len(batch_generator)}")

## 14. Contrastive Training with Bucketed Sampling

In [ ]:
class ContrastiveLoss(tf.keras.losses.Loss):
    def __init__(self, temperature=0.2, **kwargs):
        super().__init__(**kwargs)
        self.temperature = temperature
    
    def call(self, audio_emb, text_emb):
        logits = tf.matmul(audio_emb, text_emb, transpose_b=True) / self.temperature
        batch_size = tf.shape(audio_emb)[0]
        labels = tf.range(batch_size)
        
        loss_a2t = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=labels, logits=logits)
        loss_t2a = tf.nn.sparse_softmax_cross_entropy_with_logits(labels=labels, logits=tf.transpose(logits))
        
        return (tf.reduce_mean(loss_a2t) + tf.reduce_mean(loss_t2a)) / 2

# Setup
optimizer = tf.keras.optimizers.Adam(LEARNING_RATE)
loss_fn = ContrastiveLoss(temperature=TEMPERATURE)

@tf.function
def train_step(audio_batch, text_batch):
    with tf.GradientTape() as tape:
        audio_proj = audio_projection(audio_batch, training=True)
        text_proj = text_projection(text_batch, training=True)
        loss = loss_fn(audio_proj, text_proj)
    
    variables = audio_projection.trainable_variables + text_projection.trainable_variables
    gradients = tape.gradient(loss, variables)
    optimizer.apply_gradients(zip(gradients, variables))
    return loss

print("Training setup complete")

In [ ]:
# Training loop
print(f"Training for {EPOCHS} epochs with bucketed sampling...")
print(f"  Batch size: {batch_generator.batch_size}")
print(f"  Batches per epoch: {len(batch_generator)}")

history = {'loss': []}
best_loss = float('inf')

for epoch in range(EPOCHS):
    epoch_losses = []
    
    for audio_batch, text_batch in batch_generator.generate_epoch():
        audio_batch = tf.constant(audio_batch, dtype=tf.float32)
        text_batch = tf.constant(text_batch, dtype=tf.float32)
        
        loss = train_step(audio_batch, text_batch)
        epoch_losses.append(loss.numpy())
    
    avg_loss = np.mean(epoch_losses)
    history['loss'].append(avg_loss)
    
    # Learning rate decay
    if epoch > 0 and epoch % 50 == 0:
        if avg_loss >= best_loss:
            new_lr = optimizer.learning_rate.numpy() * 0.5
            if new_lr >= 1e-6:
                optimizer.learning_rate.assign(new_lr)
                print(f"\nEpoch {epoch}: LR reduced to {new_lr:.2e}")
    
    if avg_loss < best_loss:
        best_loss = avg_loss
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}/{EPOCHS} - Loss: {avg_loss:.4f} - Best: {best_loss:.4f}")

print(f"\nTraining complete!")
print(f"Final loss: {history['loss'][-1]:.4f}")
print(f"Best loss: {best_loss:.4f}")

Training for 3000 epochs with bucketed sampling...
  Batch size: 32
  Batches per epoch: 984
Epoch 10/3000 - Loss: 1.4729 - Best: 1.4715
Epoch 40/3000 - Loss: 1.4783 - Best: 1.4715
Epoch 50/3000 - Loss: 1.4791 - Best: 1.4675
Epoch 80/3000 - Loss: 1.4753 - Best: 1.4675
Epoch 90/3000 - Loss: 1.4735 - Best: 1.4675
Epoch 110/3000 - Loss: 1.4795 - Best: 1.4675
Epoch 120/3000 - Loss: 1.4862 - Best: 1.4675
Epoch 190/3000 - Loss: 1.4800 - Best: 1.4642
Epoch 200/3000 - Loss: 1.4705 - Best: 1.4642
Epoch 290/3000 - Loss: 1.4769 - Best: 1.4642
Epoch 330/3000 - Loss: 1.4735 - Best: 1.4642
Epoch 370/3000 - Loss: 1.4771 - Best: 1.4630
Epoch 380/3000 - Loss: 1.4726 - Best: 1.4630
Epoch 410/3000 - Loss: 1.4778 - Best: 1.4630
Epoch 420/3000 - Loss: 1.4791 - Best: 1.4630
Epoch 430/3000 - Loss: 1.4681 - Best: 1.4630
Epoch 440/3000 - Loss: 1.4599 - Best: 1.4599
Epoch 460/3000 - Loss: 1.4697 - Best: 1.4599
Epoch 470/3000 - Loss: 1.4721 - Best: 1.4599
Epoch 590/3000 - Loss: 1.4725 - Best: 1.4599


In [ ]:
# Plot training history
plt.figure(figsize=(12, 5))
plt.plot(history['loss'])
plt.title('Contrastive Loss (Bucketed Sampling)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.axhline(y=1.0, color='r', linestyle='--', label='Target')
plt.axhline(y=0.5, color='g', linestyle='--', label='Excellent')
plt.legend()
plt.savefig(str(MODEL_DIR / 'training_loss.png'), dpi=150)
plt.show()

## 15. Evaluation

In [ ]:
# Project embeddings
audio_projected = audio_projection.predict(audio_embeddings)
text_projected = text_projection.predict(text_embeddings)

# Compute similarity matrix (sample for visualization)
sample_size = min(500, len(audio_projected))
similarity_matrix = cosine_similarity(
    audio_projected[:sample_size],
    text_projected[:sample_size]
)

# Accuracy (match to same bucket, not exact index)
predictions = np.argmax(similarity_matrix, axis=1)
pred_buckets = bucket_ids[:sample_size][predictions]
true_buckets = bucket_ids[:sample_size]
bucket_accuracy = np.mean(pred_buckets == true_buckets)

print(f"\n{'='*60}")
print(f"EVALUATION RESULTS")
print(f"{'='*60}")
print(f"Bucket Matching Accuracy: {bucket_accuracy*100:.2f}%")
print(f"Mean diagonal similarity: {np.mean(np.diag(similarity_matrix)):.3f}")
print(f"Mean off-diagonal: {np.mean(similarity_matrix[~np.eye(sample_size, dtype=bool)]):.3f}")
print(f"Similarity gap: {np.mean(np.diag(similarity_matrix)) - np.mean(similarity_matrix[~np.eye(sample_size, dtype=bool)]):.3f}")

## 16. Export for ESP32

In [ ]:
# Save Keras model
audio_projection.save(str(MODEL_DIR / 'audio_projection.h5'))
print(f"Saved Keras model: {MODEL_DIR / 'audio_projection.h5'}")

# Convert to TFLite with quantization
converter = tf.lite.TFLiteConverter.from_keras_model(audio_projection)
converter.optimizations = [tf.lite.Optimize.DEFAULT]

# Representative dataset for quantization
def representative_dataset():
    for i in range(min(100, len(audio_embeddings))):
        yield [audio_embeddings[i:i+1].astype(np.float32)]

converter.representative_dataset = representative_dataset
converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
converter.inference_input_type = tf.float32
converter.inference_output_type = tf.float32

tflite_model = converter.convert()

tflite_path = MODEL_DIR / 'audio_projection_quantized.tflite'
with open(tflite_path, 'wb') as f:
    f.write(tflite_model)

print(f"Quantized TFLite model: {tflite_path}")
print(f"Size: {len(tflite_model) / 1024:.2f} KB")

In [ ]:
# Generate query embeddings (one per unique question)
unique_questions = list(dict.fromkeys(questions))  # Preserve order, remove duplicates
print(f"Unique questions: {len(unique_questions)}")

# Get text embeddings for unique questions
query_text_embeddings = text_encoder.encode(unique_questions)

# Project to 256-dim
query_embeddings = text_projection.predict(query_text_embeddings)
print(f"Query embeddings shape: {query_embeddings.shape}")

# Save as binary (256 floats per query)
embeddings_path = MODEL_DIR / 'embeddings.bin'
with open(embeddings_path, 'wb') as f:
    for emb in query_embeddings:
        f.write(struct.pack(f'{EMBEDDING_DIM}f', *emb))

print(f"Query embeddings saved: {embeddings_path}")
print(f"Size: {os.path.getsize(embeddings_path) / 1024:.2f} KB")

# Save intent texts
intents_path = MODEL_DIR / 'intents.txt'
with open(intents_path, 'w') as f:
    for q in unique_questions:
        f.write(f"{q}\n")

print(f"Intent texts saved: {intents_path}")

## 17. Summary

In [ ]:
print("="*60)
print("AUDIO EMBEDDING DATASET GENERATION COMPLETE")
print("="*60)
print(f"\nDataset:")
print(f"  Total questions: {len(questions)}")
print(f"  Buckets: {len(buckets)}")
print(f"  Audio samples: {len(audio_embeddings)}")
print(f"\nModel Performance:")
print(f"  Final loss: {history['loss'][-1]:.4f}")
print(f"  Bucket accuracy: {bucket_accuracy*100:.2f}%")
print(f"\nOutput Files:")
for f in MODEL_DIR.glob('*'):
    size = os.path.getsize(f)
    if size > 1024*1024:
        print(f"  {f.name}: {size/1024/1024:.2f} MB")
    else:
        print(f"  {f.name}: {size/1024:.1f} KB")

print(f"\nFor ESP32 deployment:")
print(f"  Copy models/audio_projection_quantized.tflite -> SD:/models/projection.tflite")
print(f"  Copy models/embeddings.bin -> SD:/data/embeddings.bin")
print(f"  Copy models/intents.txt -> SD:/data/intents.txt")